# 02 – Praproses, ROI, dan split (CP2, FR-02 sampai FR-04)

Lima langkah praproses → cache memmap uint8 (256×256, ber-hash) + peta ROI Frangi. Split per pasien 5 lipatan × 2 ulangan, 15% pasien latih sebagai validasi; split S7. Uji kebocoran L1 sampai L3 dan L5.

Normalisasi ImageNet tetap (diterapkan saat dibaca), tidak ada citra dibuang, parameter Frangi dikunci setelah pemeriksaan visual.

In [ ]:
import json, os, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

def _root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'configs' / 'base.yaml').exists():
            return q
    raise FileNotFoundError('configs/base.yaml tidak ditemukan')
ROOT = _root(); os.chdir(ROOT); sys.path.insert(0, str(ROOT))

from src.config import load_merged, P
from src.launch import run_module

CFG = 'configs/base.yaml'
cfg = load_merged(CFG)
DEVICES = None      # mis. [0, 1]: satu rantai job per GPU lewat CUDA_VISIBLE_DEVICES (PRD §8)
REPS, FOLDS = None, None   # None = semua. Membatasi JOB YANG DILUNCURKAN saja, bukan memilih hasil; gerbang butuh semuanya.


## 1. Praproses dan cache (idempoten; dilewati bila cache ber-hash sama sudah DONE)

In [ ]:
rc = run_module('src.preprocess', ['--config', CFG])
print('kode keluar:', rc)

## 2. Split per pasien (dan S7 bila audit menyatakan memenuhi syarat)

In [ ]:
s7_ok = json.loads(P(cfg, 'cp1', 'audit_summary.json').read_text())['s7'].get('s7_dapat_dijalankan')
rc = run_module('src.splits', ['--config', CFG] + (['--s7'] if s7_ok else []))
print('S7 dapat dijalankan:', s7_ok, '| kode keluar:', rc)
display(pd.read_csv(P(cfg, 'splits', 'fold_summary.csv')))

## 3. Pemeriksaan visual ROI (FR-03)

Periksa gambar. Bila ROI wajar: `python -m src.approve roi --by NAMA` (mengunci parameter dan menandai). Bila tidak, ubah parameter di `configs/base.yaml` lalu ulangi; **jangan** memilih parameter berdasarkan hasil klasifikasi.

In [ ]:
d = P(cfg, 'cp2', 'roi_check')
for f in sorted(d.glob('roi_check_*.png'))[:2]:
    display(Image(filename=str(f)))
print('Semua halaman ada di', d)
display(pd.read_csv(d / 'roi_examples_patients.csv').head(10))

## 4. Pemeriksaan checkpoint CP2 dan `manifest.json` (sel terakhir)

In [ ]:
from src.checkpoint import finalize_cp2
report = finalize_cp2(cfg)
assert report['checkpoint'] == 'CP2'